In [ ]:
'''
    GDSC
'''
#载入数据
train_data_dir = "../../MOCO/data/zscore_train_Depmap_ProteinCodingGene_mRNA.npz"  # Train data
test_data_dir = "../../MOCO/data/zscore_test_Depmap_ProteinCodingGene_mRNA.npz"    # Test data
output_path = '../../data/SHAP/Cell_out'                                           # SHAP cell to mid output dir 
save_path = "../../MOCO/result/mRNA_protein/checkpoint_0199.pth.tar"               # MOCO model

In [1]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.multiprocessing as mp
import sys

sys.path.append("../../MOCO")
sys.path.append("../../MOCO/utils")
os.chdir("../../MOCO")

from  utils import *
from builder import *
from dataloader import *
from densenet import *
from embedding_extractor_test import *
from preprocessing import *
from utils import *
from parse import *
import densenet2 as models

from builder import *
from parse import *

/mnt/hpc/home/shilei/miniconda3/envs/nrf-llama2/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/hpc/home/shilei/miniconda3/envs/nrf-llama2/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# 加载数据
class MoCoDataset_test(torch.utils.data.Dataset):
    def __init__(self, filepath, transform = None):
        self.path = filepath
        self.transform = transform

    def __len__(self):
        self.data = np.load(self.path,allow_pickle=True)
        return self.data['x'].shape[0]   
        
    def __getitem__(self,i):
        
        x = self.data['x'][i]
        label = self.data['y'][i]
        y = labe_data[labe_data["Patient_ID"] == label]["Cancer"].values[0]
        return x, y
    
class MoCoDataset_test1(torch.utils.data.Dataset):
    def __init__(self, filepath, transform = None):
        self.path = filepath
        self.transform = transform

    def __len__(self):
        self.data = np.load(self.path,allow_pickle=True)
        return self.data['x'].shape[0]   
        
    def __getitem__(self,i):
        
        x = self.data['x'][i]
        label = self.data['y'][i]
        #gene_ls = self.data['z']
        #y = labe_data[labe_data["Patient_ID"] == label]["Cancer"].values[0]
        return x, label#, gene_ls

def load_para(save_path):
    checkpoint = torch.load(save_path)
    #载入模型参数
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:]  # remove 'module.' of dataparallel
        if name.endswith(".weight") or name.endswith(".bias"):
            if name.split(".")[2] in ["0","2"]:
                name = f'{name.split(".")[0]}.{name.split(".")[1]}.{name.split(".")[3]}'
        new_state_dict[name] = v
    return new_state_dict


In [3]:
args = moco_parser().parse_args('')
#args
args = moco_parser().parse_args('')
model_dict = models.__dict__

kwargs = {"randomzero" : 0.3}
model = MoCo(
        model_dict["densenet27"],
        num_batches = 32, 
        in_features = args.in_features, 
        dim=args.moco_dim, 
        K=args.moco_k, 
        m=args.moco_m, 
        T=args.moco_t, 
        mlp=args.mlp,
        **kwargs)

train_dataset = MoCoDataset_test1(train_data_dir)
test_dataset = MoCoDataset_test1(test_data_dir)


if not os.path.exists(output_path):
    os.makedirs(output_path)


#载入checkpoint文件
#save_path = "/mnt/hpc/home/shilei/RongfangNie/project/Nierongfang/TumorDrugPredict/code/moco-main copy 2/model/checkpoint_0041.pth copy.tar"

model.load_state_dict(load_para(save_path))
model.eval()

test_df = np.load(test_data_dir,allow_pickle=True)
gene_ls = test_df['z']

device = "cuda:0"
model.to(device)

MoCo(
  (encoder_q): DenseNet(
    (drop1): Dropout(p=0.3, inplace=False)
    (features): Sequential(
      (conv0): Linear(in_features=10000, out_features=7168, bias=False)
      (norm0): GroupNorm(1, 7168, eps=1e-05, affine=True)
      (relu0): ReLU()
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): GroupNorm(1, 7168, eps=1e-05, affine=True)
          (relu1): ReLU()
          (conv1): Linear(in_features=7168, out_features=4096, bias=False)
          (norm2): GroupNorm(1, 4096, eps=1e-05, affine=True)
          (relu2): ReLU()
          (conv2): Linear(in_features=4096, out_features=1024, bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): GroupNorm(1, 8192, eps=1e-05, affine=True)
          (relu1): ReLU()
          (conv1): Linear(in_features=8192, out_features=4096, bias=False)
          (norm2): GroupNorm(1, 4096, eps=1e-05, affine=True)
          (relu2): ReLU()
          (conv2): Linear(in_features=4096, out_featu

In [5]:
train_dataloader = DataLoader(train_dataset,  batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=1)
test_dataloader1 = DataLoader(train_dataset, batch_size=1)

In [6]:
for train_batch in train_dataloader:
    train_data,label = train_batch
    break

In [7]:
def example_model_predict(data):
    X = []
    q = torch.tensor(data,dtype=torch.float32).to(device)
    #q = q.unsqueeze(0)
    #frature = q.detach().numpy()[0]
    frature = model.encoder_q(q).cpu().detach().numpy()
    X = X + frature.tolist()
    X = np.array(X)
    return X

In [10]:
import shap
import numpy as np

model_predict = example_model_predict # 使用上面的示例黑盒模型
model_encoder = model.encoder_q
model_encoder.to(device)
model_encoder.eval()

explainer = shap.DeepExplainer(model_encoder, torch.tensor(train_data,dtype=torch.float32).to(device))

/tmp/ipykernel_103339/1467729589.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  explainer = shap.DeepExplainer(model_encoder, torch.tensor(train_data,dtype=torch.float32).to(device))


In [ ]:
import shap
import numpy as np

model_predict = example_model_predict # 使用上面的示例黑盒模型


for test_batch in test_dataloader:
    test_data,label = test_batch
    if os.path.exists(f"{output_path}/{label[0]}_shap_values.csv"):
        continue
    #break
    #X_explain =  test_data.cpu().numpy()
    X_explain =  torch.tensor(test_data,dtype=torch.float32).to(device)
    #explainer = shap.KernelExplainer(model_predict, background_data, link="identity")
    shap_values = explainer.shap_values(X_explain[0:1], check_additivity=False) # 只解释第一个样本，nsamples设得很小
    df = pd.DataFrame(shap_values[0])
    df.index = gene_ls
    df.columns = explainer.expected_value
    df.to_csv(f"{output_path}/{label[0]}_shap_values.csv")

for test_batch in test_dataloader1:
    test_data,label = test_batch
    if os.path.exists(f"{output_path}/{label[0]}_shap_values.csv"):
        continue
    #break
    #X_explain =  test_data.cpu().numpy()
    X_explain =  torch.tensor(test_data,dtype=torch.float32).to(device)
    #explainer = shap.KernelExplainer(model_predict, background_data, link="identity")
    shap_values = explainer.shap_values(X_explain[0:1], check_additivity=False) # 只解释第一个样本，nsamples设得很小
    df = pd.DataFrame(shap_values[0])
    df.index = gene_ls
    df.columns = explainer.expected_value
    df.to_csv(f"{output_path}/{label[0]}_shap_values.csv")